# Tool Use Loop — How the conversation history works

This notebook shows the full multi-turn tool use loop step by step.
You will see every API call, every stop_reason, and every message going in and out.

We use a simple example first (nothing to do with Pattern Mirror),
then show the real Pattern Mirror flow at the end.

In [4]:
import os, json
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()
client = Anthropic(api_key=os.environ.get("ANTHROPIC_API_KEY"))

## Part 1 — A simple tool Claude will want to call

We define a `get_weather` tool. Claude can't actually check the weather,
so when we ask it, it will call the tool instead of answering directly.
We'll fake the result and send it back.

In [5]:
WEATHER_TOOL = {
    "name": "get_weather",
    "description": "Get the current weather for a city.",
    "input_schema": {
        "type": "object",
        "properties": {
            "city": {"type": "string", "description": "The city name"}
        },
        "required": ["city"]
    }
}

def fake_weather(city: str) -> str:
    """Pretend we fetched real weather data."""
    return f"Weather in {city}: 22°C, partly cloudy."

## Call 1 — Send the question, watch what comes back

In [16]:
messages = [{"role": "user", "content": "What's the weather like in London right now?"}]

response_1 = client.messages.create(
    model="claude-haiku-4-5-20251001",
    max_tokens=300,
    tools=[WEATHER_TOOL],
    messages=messages
)

print("=== CALL 1 ===")
print(response_1)
print(f"stop_reason: {response_1.stop_reason}")
print(f"content blocks: {len(response_1.content)}")
print()
for block in response_1.content:
    print(f"block type: {block.type}")
    if block.type == "tool_use":
        print(f"  tool name: {block.name}")
        print(f"  tool id:   {block.id}")
        print(f"  input:     {block.input}")
    elif block.type == "text":
        print(f"  text: {block.text}")

=== CALL 1 ===
Message(id='msg_01BAyEghxVf5tznptkGcPsVB', container=None, content=[ToolUseBlock(id='toolu_01KFopZtDAVUxPVLcWV6x8S2', caller=DirectCaller(type='direct'), input={'city': 'London'}, name='get_weather', type='tool_use')], model='claude-haiku-4-5-20251001', role='assistant', stop_details=None, stop_reason='tool_use', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=576, output_tokens=54, server_tool_use=None, service_tier='standard'))
stop_reason: tool_use
content blocks: 1

block type: tool_use
  tool name: get_weather
  tool id:   toolu_01KFopZtDAVUxPVLcWV6x8S2
  input:     {'city': 'London'}


**What you should see:**
- `stop_reason: tool_use` — Claude stopped because it wants to call a tool
- A `tool_use` block with `name: get_weather` and `input: {"city": "London"}`
- No text — Claude didn't answer yet, it's waiting for the tool result

Now we execute the tool ourselves and build the message history.

In [17]:
# Extract the tool call
tool_block = response_1.content[0]  # the tool_use block
city = tool_block.input["city"]

# Execute the tool — we run this, not Claude
weather_result = fake_weather(city)
print(f"Tool result: {weather_result}")
print()

# Build the message history:
# 1. Add Claude's response (the tool call) as the assistant turn
messages.append({"role": "assistant", "content": response_1.content})

# 2. Add our tool result as the user turn
messages.append({
    "role": "user",
    "content": [{
        "type": "tool_result",
        "tool_use_id": tool_block.id,  # must match the tool call id
        "content": weather_result
    }]
})

print("=== MESSAGE HISTORY SO FAR ===")
print(messages)
for i, msg in enumerate(messages):
    role = msg["role"]
    content = msg["content"]
    if isinstance(content, str):
        print(f"[{i}] {role}: {content}")
    elif isinstance(content, list):
        for block in content:
            if hasattr(block, 'type'):
                print(f"[{i}] {role}: [{block.type}] {getattr(block, 'name', '')} {getattr(block, 'input', '')}")
            elif isinstance(block, dict):
                print(f"[{i}] {role}: [{block.get('type')}] {block.get('content', '')}")

Tool result: Weather in London: 22°C, partly cloudy.

=== MESSAGE HISTORY SO FAR ===
[{'role': 'user', 'content': "What's the weather like in London right now?"}, {'role': 'assistant', 'content': [ToolUseBlock(id='toolu_01KFopZtDAVUxPVLcWV6x8S2', caller=DirectCaller(type='direct'), input={'city': 'London'}, name='get_weather', type='tool_use')]}, {'role': 'user', 'content': [{'type': 'tool_result', 'tool_use_id': 'toolu_01KFopZtDAVUxPVLcWV6x8S2', 'content': 'Weather in London: 22°C, partly cloudy.'}]}]
[0] user: What's the weather like in London right now?
[1] assistant: [tool_use] get_weather {'city': 'London'}
[2] user: [tool_result] Weather in London: 22°C, partly cloudy.


**What you should see in the history:**
- Turn 0 (user): the original question
- Turn 1 (assistant): Claude's tool_use call
- Turn 2 (user): our tool_result with the weather data

Now Call 2 — Claude reads the full history and answers.

In [10]:
response_2 = client.messages.create(
    model="claude-haiku-4-5-20251001",
    max_tokens=300,
    tools=[WEATHER_TOOL],
    messages=messages
)

print("=== CALL 2 ===")
print(f"stop_reason: {response_2.stop_reason}")
print()
for block in response_2.content:
    print(f"block type: {block.type}")
    if block.type == "text":
        print(f"  text: {block.text}")

=== CALL 2 ===
stop_reason: end_turn

block type: text
  text: The weather in London right now is **22°C (about 72°F) and partly cloudy**. It's a pleasant day!


**What you should see:**
- `stop_reason: end_turn` — Claude is done, no more tool calls
- A `text` block with the actual weather answer, using the data we provided

---

## Part 2 — The Pattern Mirror flow

Same loop, but now:
- Claude can call `lookup_framework` (optional, its choice)
- Claude must call `pattern_analysis` at the end (forced)
- We print every step so you can see the full flow

In [18]:
import sys
sys.path.insert(0, ".")

from prompts import SYSTEM_PROMPT, ANALYSIS_TOOL
from tools import LOOKUP_FRAMEWORK_TOOL, handle_tool_call
from validator import format_answers

# Set 3 — conflict avoidance, most likely to trigger fawn_response lookup
MOCK_ANSWERS = [
    {"question": "Q1", "answer": "I told myself it was probably my fault for not communicating clearly enough. Even though I'm not sure it was."},
    {"question": "Q2", "answer": "Someone spoke over me repeatedly in a meeting. I thought — I should have been clearer, maybe my point wasn't worth making anyway."},
    {"question": "Q3", "answer": "That I'm angry. I'm very agreeable on the outside. If people saw how much resentment I carry they'd think I'm ungrateful or difficult."},
    {"question": "Q4", "answer": "Confrontation. Two days ago when I needed to push back on something but smiled and said it was fine."},
    {"question": "Q5", "answer": "I'm just not someone who makes things difficult for others."},
    {"question": "Q6", "answer": "Yes — at home, with friends, everywhere. I smooth things over and then disappear to be angry alone."}
]

tools = [LOOKUP_FRAMEWORK_TOOL, ANALYSIS_TOOL]
messages = [{"role": "user", "content": format_answers(MOCK_ANSWERS)}]

print("Starting tool use loop...\n")

Starting tool use loop...



In [19]:
# === CALL 1 — Claude reads answers, decides what to do ===
response = client.messages.create(
    model="claude-sonnet-4-5",
    max_tokens=2048,
    system=SYSTEM_PROMPT,
    tools=tools,
    messages=messages
)

print("=== CALL 1 ===")
print(f"stop_reason: {response.stop_reason}")
print(f"content blocks: {len(response.content)}")
print()
for block in response.content:
    print(f"  block type: {block.type}")
    if block.type == "tool_use":
        print(f"  tool name:  {block.name}")
        print(f"  tool id:    {block.id}")
        print(f"  input:      {block.input}")
    elif block.type == "text":
        print(f"  text: {block.text[:200]}")

=== CALL 1 ===
stop_reason: tool_use
content blocks: 1

  block type: tool_use
  tool name:  lookup_framework
  tool id:    toolu_01VAWiLVDkHF57KkbNpmfDBa
  input:      {'framework_key': 'fawn_response'}


In [20]:
# === Execute any tool calls and build history ===
tool_calls = [b for b in response.content if b.type == "tool_use"]

if tool_calls:
    # Add Claude's response to history
    messages.append({"role": "assistant", "content": response.content})

    tool_results = []
    for block in tool_calls:
        print(f"Executing tool: {block.name}({block.input})")
        result = handle_tool_call(block.name, block.input)
        print(f"Result:\n{result}\n")
        tool_results.append({
            "type": "tool_result",
            "tool_use_id": block.id,
            "content": result
        })

    # Send results back
    messages.append({"role": "user", "content": tool_results})
    print(f"Message history now has {len(messages)} turns")
else:
    print("Claude called no tools on Call 1 — went straight to pattern_analysis")

Executing tool: lookup_framework({'framework_key': 'fawn_response'})
Result:
Framework: Fawn Response (Trauma-Informed / Polyvagal)
Description: The fawn response is a survival strategy in which the person appeases, agrees, and accommodates to avoid conflict or threat. It is a nervous system response, not a personality trait — the body learned that compliance reduces danger. Over time it becomes automatic, operating even in low-stakes situations where no real threat exists.
Protocol note: The fawn response bypasses conscious choice — it fires before the person decides to agree. Detection must happen in the body, before the words come out.
Pattern tags: people-pleasing, conflict-avoidance, self-erasure, compliance, resentment

Message history now has 3 turns


In [23]:
# === CALL 2 — Force pattern_analysis with full history ===
response_final = client.messages.create(
    model="claude-sonnet-4-5",
    max_tokens=2048,
    system=SYSTEM_PROMPT,
    tools=tools,
    tool_choice={"type": "tool", "name": "pattern_analysis"},
    messages=messages
)

print("=== CALL 2 (forced pattern_analysis) ===")
print(response_final)
print(f"stop_reason: {response_final.stop_reason}")
print()

for block in response_final.content:
    if block.type == "tool_use" and block.name == "pattern_analysis":
        result = block.input
        print("pattern_analysis output:")
        print(json.dumps(result, indent=2))

=== CALL 2 (forced pattern_analysis) ===
Message(id='msg_01GEt85ofB3W3Lge1hehbqEj', container=None, content=[ToolUseBlock(id='toolu_017wLDnw8Rrpqtadcua5ymzh', caller=DirectCaller(type='direct'), input={'core_pattern': {'name': 'Preemptive Self-Erasure', 'description': "A reflexive response that converts your needs, anger, and boundaries into fault, vagueness, or accommodation before anyone else can reject them. It operates as a body-level survival strategy — compliance fires before conscious choice. It protects you by ensuring you are never the person who 'makes things difficult,' but the cost is a split: an agreeable exterior and a private reservoir of resentment that no one sees."}, 'evidence': ["I told myself it was probably my fault for not communicating clearly enough. Even though I'm not sure it was.", "If people saw how much resentment I carry they'd think I'm ungrateful or difficult.", 'I smooth things over and then disappear to be angry alone.'], 'domains': ['Work (meetings, c

In [22]:
# === Access individual fields from the structured output ===
print("Core pattern: ", result["core_pattern"]["name"])
print("Payoff:       ", result["payoff"])
print()
print("Protocol detection trigger:")
print(" ", result["protocol"]["detection_trigger"])
print()
print("Fallback if overwhelmed:")
print(" ", result["protocol"]["fallback_shutdown"])

Core pattern:  Preemptive Self-Blame
Payoff:        It makes sure you are never seen as difficult, ungrateful, or angry. It keeps you safe from confrontation and from the possibility that conflict could damage connection.

Protocol detection trigger:
  The moment you feel the impulse to smile, nod, or say 'it's fine' when something isn't. Physically: tightness in your throat or chest, the sensation of words stopping before they reach your mouth.

Fallback if overwhelmed:
  Leave the room or end the conversation without explanation. Do not process, do not message anyone. Sit or lie down and place both hands flat on your stomach.
